# DSA Drill - Find Median from Data Stream (LC 295 - Hard)

> Design `MedianFinder`: `add_num(x)` adds an integer; `find_median()` returns the median of all numbers added so far. If the count is even, return the average of the two middle elements.

**Examples**
- add 1, add 2 -> median 1.5 · add 3 -> median 2 · add 4 -> median 2.5

**Constraints:** values can be negative; up to 5*10^4 calls.

**The two-heap invariant (verbalize this cold):** a **max-heap** `lo` holds the smaller half; a **min-heap** `hi` holds the larger half; keep `len(lo) == len(hi)` or `len(lo) == len(hi) + 1`. Then:
- odd total -> median = top of `lo`
- even total -> median = (top of lo + top of hi) / 2

**The invariant-preserving dance on every add:** always push into one heap then rebalance by moving its top into the other **only when the size invariant breaks AND the order invariant breaks** - the classic implementation: push to lo, then move lo's max into hi if lo grew too big; but also must ensure lo's max <= hi's min. State the two invariants and how each add preserves both.

**Landmines:**
1. Why TWO heaps and not one sorted list (O(n) insert) or one heap?
2. The subtle case: adding an element SMALLER than lo's max when lo is full - trace it.
3. What breaks if you let len(lo) = len(hi) + 2?

One hint allowed.

## Intuition (plain): the seesaw

Keep the numbers on a seesaw: the **smaller half** on the left (max-heap so its biggest element is on top), the **larger half** on the right (min-heap so its smallest is on top). The median is always where the two halves touch: left top (odd) or the average of both tops (even).

Every add either lands on the correct side and keeps `|left| - right| in {0,1}`, or one element crosses over to rebalance. The dance:

```text
add x:
  push x to lo;  pop max of lo -> push to hi        # x transits through lo
  if len(hi) > len(lo): pop min of hi -> push to lo  # keep left >= right
median: lo.top if len(lo) > len(hi) else (lo.top + hi.top) / 2
```

The transit trick handles the "x is smaller than everything" case for free: x goes into lo and gets evicted immediately, leaving the real max behind.

In [ ]:
import heapq

class MedianFinder:
    """Two-heap median: add O(log n), median O(1). lo = max-heap (smaller half),
    hi = min-heap (larger half); invariants after every add:
      1. order:   max(lo) <= min(hi)
      2. size:    len(lo) == len(hi) or len(lo) == len(hi) + 1
    The dance: push to lo, ALWAYS transit lo's max to hi, then rebalance hi -> lo
    if hi grew bigger. The unconditional transit handles the "x smaller than
    everything" case for free (x enters lo and is evicted immediately)."""

    def __init__(self) -> None:
        self.lo: list[int] = []   # negated values -> max-heap
        self.hi: list[int] = []   # real values -> min-heap

    def add_num(self, num: int) -> None:
        heapq.heappush(self.lo, -num)
        heapq.heappush(self.hi, -heapq.heappop(self.lo))   # ALWAYS transit (the bug fix)
        if len(self.hi) > len(self.lo):
            heapq.heappush(self.lo, -heapq.heappop(self.hi))

    def find_median(self) -> float:
        if len(self.lo) > len(self.hi):
            return float(-self.lo[0])
        return (-self.lo[0] + self.hi[0]) / 2.0

In [ ]:
def _brute_median(arr):
    s = sorted(arr)
    n = len(s)
    return float(s[n // 2]) if n % 2 else (s[n//2 - 1] + s[n//2]) / 2.0

mf = MedianFinder()
seqs = [[1], [1, 2], [1, 2, 3], [2, 1, 5, 7, 2, 0, 5]]
for arr in seqs:
    m = MedianFinder()
    for x in arr: m.add_num(x)
    assert m.find_median() == _brute_median(arr), (arr, m.find_median(), _brute_median(arr))

import random
random.seed(29)
for trial in range(150):
    n = random.randint(1, 120)
    arr = [random.randint(-50, 50) for _ in range(n)]
    m = MedianFinder()
    for i, x in enumerate(arr, 1):
        m.add_num(x)
        assert m.find_median() == _brute_median(arr[:i]), (arr[:i], m.find_median(), _brute_median(arr[:i]))
print("examples + 150 incremental streams (every prefix checked) == brute force")

In [ ]:
# Edge cases - run me, say WHY out loud
m = MedianFinder(); m.add_num(-5); assert m.find_median() == -5.0      # single negative
m = MedianFinder()
for x in [3, 1, 4, 1, 5, 9, 2, 6]: m.add_num(x)
assert m.find_median() == 3.5                                          # even count, decimals
m = MedianFinder()
for x in range(100): m.add_num(x)                                      # strictly increasing
assert m.find_median() == 49.5
print("edge cases pass OK")

## Post-mortem (fill in your own words)
- Sorted-list approach complexity (why rejected):
- The two invariants and the dance that preserves both:
- Trace [3, 1, 4] step by step through your implementation:
- Complexity: add / median:
- One-line pattern (copy into dsa/patterns.md):